# model_strict_adapters_v1

验证 `logistic_clf`、`ridge_mse`、`ridge_ic` 从通用 smoke fallback 进入显式 adapter 后，训练、预测、artifact metadata 和 manifest 汇总是否仍保持统一框架契约。

## 实验治理

- Baseline：旧 `smoke_adapter` 通用训练路径。
- Direct comparison：`ridge_mse` 的 MSE 目标 vs `ridge_ic` 的 rank/IC proxy 目标。
- Role：diagnose，验证高优先级线性模型 adapter 替换是否进入稳定 `src/model` 接口。
- Target window：2026-01-01 到 2026-04-30，2026-03-31 前训练，之后测试。
- Changed layer：model。
- External reference：QuantBench 模型矩阵中 logistic/ridge family 的显式监督模型入口。
- Acceptance：三类模型输出 `paper_strict_adapter`；artifact metadata 记录 adapter 名称；`ridge_ic` 预测不同于 `ridge_mse`。

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

repo_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'src').exists() and (path / 'AGENTS.md').exists()
)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.data import build_supervised_dataset, load_soxl_support_frame
from src.model import fit_model_artifact, load_model_artifact, save_model_artifact
from src.model.training import fit_predict_model

CARD = repo_root / 'research' / 'model_strict_adapters_v1'
CARD.mkdir(parents=True, exist_ok=True)

frame = load_soxl_support_frame(start='2026-01-01', end='2026-04-30')
dataset = build_supervised_dataset(frame, horizon=1)
train = dataset.frame.loc[dataset.frame['date'] <= '2026-03-31'].copy()
test = dataset.frame.loc[dataset.frame['date'] > '2026-03-31'].copy()
feature_columns = dataset.feature_columns[:6]
{
    'train_rows': int(len(train)),
    'test_rows': int(len(test)),
    'feature_columns': list(feature_columns),
    'train_end': str(train['date'].max().date()),
    'test_start': str(test['date'].min().date()),
    'test_end': str(test['date'].max().date()),
}


{'train_rows': 61,
 'test_rows': 20,
 'feature_columns': ['soxl_return_1d',
  'soxx_return_1d',
  'soxl_return_5d',
  'soxx_return_5d',
  'soxl_soxx_relative_5d',
  'vix_level'],
 'train_end': '2026-03-31',
 'test_start': '2026-04-01',
 'test_end': '2026-04-29'}

In [2]:
model_names = ('logistic_clf', 'ridge_mse', 'ridge_ic')
predictions = pd.concat(
    [fit_predict_model(name, train=train, test=test, feature_columns=feature_columns) for name in model_names],
    ignore_index=True,
)
predictions.to_csv(CARD / 'adapter_predictions.csv', index=False)

group_columns = ['model', 'objective', 'family', 'adapter_status', 'model_adapter']
adapter_rows = []
for keys, group in predictions.groupby(group_columns, dropna=False):
    adapter_rows.append(
        {
            **dict(zip(group_columns, keys)),
            'rows': int(len(group)),
            'prediction_mean': float(group['prediction'].mean()),
            'prediction_std': float(group['prediction'].std(ddof=0)),
            'target_corr': float(group['prediction'].corr(group['target_return'])),
        }
    )
adapter_table = pd.DataFrame(adapter_rows)
adapter_table.to_csv(CARD / 'adapter_summary.csv', index=False)

ridge_compare = predictions.pivot(index='date', columns='model', values='prediction').reset_index()
ridge_compare['ridge_ic_minus_ridge_mse'] = ridge_compare['ridge_ic'] - ridge_compare['ridge_mse']
ridge_compare.to_csv(CARD / 'ridge_ic_vs_mse.csv', index=False)

assert predictions['adapter_status'].unique().tolist() == ['paper_strict_adapter']
assert set(predictions['model_adapter']) == {'logistic_direction_v1', 'ridge_mse_v1', 'ridge_rank_ic_proxy_v1'}
assert not ridge_compare['ridge_ic'].round(10).equals(ridge_compare['ridge_mse'].round(10))
adapter_table


,model,objective,family,adapter_status,model_adapter,rows,prediction_mean,prediction_std,target_corr
0,logistic_clf,classification,linear,paper_strict_adapter,logistic_direction_v1,20,-0.023997,0.060106,0.056622
1,ridge_ic,ic,linear,paper_strict_adapter,ridge_rank_ic_proxy_v1,20,-0.461366,0.253989,-0.108419
2,ridge_mse,mse,linear,paper_strict_adapter,ridge_mse_v1,20,-0.034585,0.019276,-0.123791


In [3]:
artifact = fit_model_artifact('ridge_ic', train=train, feature_columns=feature_columns)
artifact_path = save_model_artifact(CARD / 'ridge_ic_artifact.joblib', artifact)
loaded = load_model_artifact(artifact_path)
artifact_predictions = loaded.predict(test)
artifact_predictions.to_csv(CARD / 'artifact_predictions.csv', index=False)
artifact_metadata = loaded.metadata_payload()
(CARD / 'artifact_metadata.json').write_text(json.dumps(artifact_metadata, ensure_ascii=False, indent=2), encoding='utf-8')

assert loaded.adapter_status == 'paper_strict_adapter'
assert artifact_metadata['model_adapter'] == 'ridge_rank_ic_proxy_v1'
assert artifact_predictions['model_adapter'].unique().tolist() == ['ridge_rank_ic_proxy_v1']
{
    'artifact_path': str(artifact_path),
    'metadata': artifact_metadata,
    'prediction_rows': int(len(artifact_predictions)),
}


{'artifact_path': '/Users/rziiiii/dev/workspace/research/model_strict_adapters_v1/ridge_ic_artifact.joblib',
 'metadata': {'schema_version': 'model_artifact.v1',
  'model_name': 'ridge_ic',
  'feature_columns': ['soxl_return_1d',
   'soxx_return_1d',
   'soxl_return_5d',
   'soxx_return_5d',
   'soxl_soxx_relative_5d',
   'vix_level'],
  'objective': 'ic',
  'family': 'linear',
  'adapter_status': 'paper_strict_adapter',
  'model_adapter': 'ridge_rank_ic_proxy_v1',
  'target_column': 'target_return',
  'train_rows': 61},
 'prediction_rows': 20}

In [4]:
summary = {
    'schema_version': 'model_strict_adapters_review.v1',
    'status': 'completed',
    'train_rows': int(len(train)),
    'test_rows': int(len(test)),
    'feature_columns': list(feature_columns),
    'models': list(model_names),
    'adapter_status': 'paper_strict_adapter',
    'model_adapters': sorted(predictions['model_adapter'].unique().tolist()),
    'ridge_ic_differs_from_ridge_mse': bool(
        not ridge_compare['ridge_ic'].round(10).equals(ridge_compare['ridge_mse'].round(10))
    ),
    'ridge_ic_vs_mse_abs_mean_diff': float(ridge_compare['ridge_ic_minus_ridge_mse'].abs().mean()),
    'adapter_summary_path': 'adapter_summary.csv',
    'prediction_path': 'adapter_predictions.csv',
    'artifact_path': 'ridge_ic_artifact.joblib',
    'artifact_metadata_path': 'artifact_metadata.json',
    'artifact_prediction_path': 'artifact_predictions.csv',
}
(CARD / 'analysis_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
summary


{'schema_version': 'model_strict_adapters_review.v1',
 'status': 'completed',
 'train_rows': 61,
 'test_rows': 20,
 'feature_columns': ['soxl_return_1d',
  'soxx_return_1d',
  'soxl_return_5d',
  'soxx_return_5d',
  'soxl_soxx_relative_5d',
  'vix_level'],
 'models': ['logistic_clf', 'ridge_mse', 'ridge_ic'],
 'adapter_status': 'paper_strict_adapter',
 'model_adapters': ['logistic_direction_v1',
  'ridge_mse_v1',
  'ridge_rank_ic_proxy_v1'],
 'ridge_ic_differs_from_ridge_mse': True,
 'ridge_ic_vs_mse_abs_mean_diff': 0.4367261873168422,
 'adapter_summary_path': 'adapter_summary.csv',
 'prediction_path': 'adapter_predictions.csv',
 'artifact_path': 'ridge_ic_artifact.joblib',
 'artifact_metadata_path': 'artifact_metadata.json',
 'artifact_prediction_path': 'artifact_predictions.csv'}

## 中文结论

- 做了什么：把 `logistic_clf`、`ridge_mse`、`ridge_ic` 作为第一批高优先级模型，验证它们通过 `src/model/training.py` 的显式 adapter 完成训练、预测，并通过 `src/model/artifacts.py` 保存/加载 `ridge_ic` artifact。
- 核心指标：训练样本和测试样本数量、adapter 明细、`ridge_ic` 与 `ridge_mse` 的平均绝对预测差异写入 `analysis_summary.json`；逐模型相关性写入 `adapter_summary.csv`。
- regime 差异：本轮不是完整 regime 回测；目的在于保证后续 qlib/regime 分析读取到的预测和 artifact metadata 已能区分 `model_adapter` 与 `adapter_status`。
- 是否晋升：否。本轮只把三类核心线性/分类模型从 smoke fallback 推进到框架内严格 adapter，尚不代表生产模型晋升。
- 风险：`ridge_ic` 当前是 rank/IC proxy，不是完整论文级优化器；其余深度、图、时序模型仍是 fallback，需要逐个用同一契约替换。
- 下一步：继续按 QuantBench 模型矩阵分批替换 tree/boosting/deep/graph adapter，并把每批结果接入 matrix manifest、dashboard export 和 qlib regime 分析。